*Load relevant packages:*

In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run packages.ipynb
# Local packages:
os.chdir(d['py'])
import mCGE 
os.chdir(os.path.join(d['curr'], 'py'))
import report, experiments

The file _gams_py_gdb0.gdx is still active and was not deleted.
The file _gams_py_gdb1.gdx is still active and was not deleted.
The file _gams_py_gdb2.gdx is still active and was not deleted.
The file _gams_py_gdb3.gdx is still active and was not deleted.
The file _gams_py_gdb4.gdx is still active and was not deleted.
The file _gams_py_gdb5.gdx is still active and was not deleted.
The file _gams_py_gdb6.gdx is still active and was not deleted.
The file _gams_py_gdb7.gdx is still active and was not deleted.


# Robustness checks

*Repeat analysis with variety of different elasticities. Note: We have not implemented a full routine for creating the tables. Instead, the numbers that goes into the tables are returned (with a bit of latex formatting added).*

Load base model and data:

In [2]:
t0 = 2019
v  = 'vMain' 
name = f'{v}{t0}CGE'

*Load:*

In [3]:
M = mCGE.WasteManagementCGE.load(os.path.join(d['data'], name)) # load model
ws = M.ws 
db0 = M.db.copy() # store initial solution

## 0. Baseline solution for all experiments

Initialize reporting class:

In [4]:
Rep = report.Standard(db0)

Add baseline report:

In [5]:
d0 = Rep(db0, {})
d0['db'] = db0

Empty dictionary to store solutions in:

In [6]:
baseShocks = {'Baseline': d0} # Dictionary to store shocks in

Run experiments:

In [7]:
baseShocks['IRRmetal'] = experiments.runIRR(M, Rep, baseShocks['Baseline'], m = pd.Index(['Metal'], name = 'm'), Δ=.1) # Experiment 1
baseShocks['IRRplastic'] = experiments.runIRR(M, Rep, baseShocks['Baseline'], m = pd.Index(['Plastic'], name = 'm'), Δ=.1) # Experiment 2
baseShocks['IRRall'] = experiments.runIRR(M, Rep, baseShocks['Baseline'], Δ=.1) # Experiment 3 - applies to all materials
baseShocks['RCEff'] = experiments.runRCEff(M, Rep, baseShocks['Baseline'], Δα=.1) # Experiment 4
baseShocks['taxVirgin'] = experiments.runTaxVirgin(M, Rep, baseShocks['Baseline'], Δ=.1) # experiment 5
baseShocks['RCMandate'] = experiments.runRCMandate(M, Rep, baseShocks['Baseline'], targetRate=.5, m = pd.Index(['Textile'], name = 'm')) # experiment 6
baseShocks['taxWasteGen'] = experiments.runTaxWasteGen(M, Rep, baseShocks['Baseline'], Δ=.1) # experiment 7
baseShocks['subsidyRecycledInputs'] = experiments.runSubsidyRecycledInputs(M, Rep, baseShocks['Baseline'], Δ=.1) # experiment 8

Map to experiment numbers:

In [8]:
mapExp = {1: 'IRRall', '2a': 'IRRmetal', '2b': 'IRRplastic', 3: 'RCEff', 4: 'taxVirgin', 5: 'taxWasteGen', 6: 'RCMandate', 7: 'subsidyRecycledInputs'}

### 1. Substitution between virgin/raw materials

The first robustness check investigates a higher/lower EOS for raw/virgin materials changes things:

In [9]:
m = M.get('map',m = 'P').union(M.get('map', m = 'W')) # nesting tree all relevant domestic firms

Identify relevant nodes:

In [10]:
m_ZOnm = m[m.get_level_values('n').isin('RxE_'+M.db('m'))].droplevel('nn').unique() # parent nodes in nesting tree that combines virgin and recycled

Baseline value:

In [11]:
baseσ = adj.rc_pd(db0('sigma'), m_ZOnm).mean()

Investigate effect of EOS +/- 1:

In [12]:
robCheck1 = dict.fromkeys((baseσ-1, baseσ+1)) # solution dicts
Reps1 = dict.fromkeys((baseσ-1, baseσ+1)) # reporting dicts

Solve baselines:

In [13]:
for σi in robCheck1:
    # Update parameter and re-calibrate:
    M.db.aom(pd.Series(σi, index = m_ZOnm, name = 'sigma'), priority = 'second')
    M.db.mergeInternal()
    base = M.solve(state = 'C')
    # Add new reporting function:
    Reps1[σi] = report.Standard(base)
    d0i = Reps1[σi](base, {}) # standard report
    d0i['db'] = base # add database
    robCheck1[σi] = {'Baseline': d0i} # add baseline dictionary

For each version, run through experiments and report stuff:

In [14]:
for σi in robCheck1:
    [M.db.__setitem__(k, robCheck1[σi]['Baseline']['db'][k]) for k in M.db.getTypes(['var'])]; # Revert to relevant baseline
    robCheck1[σi]['IRRmetal'] = experiments.runIRR(M, Reps1[σi], robCheck1[σi]['Baseline'], m = pd.Index(['Metal'], name = 'm'), Δ=.1) # Experiment 1
    robCheck1[σi]['IRRplastic'] = experiments.runIRR(M, Reps1[σi], robCheck1[σi]['Baseline'], m = pd.Index(['Plastic'], name = 'm'), Δ=.1) # Experiment 2
    robCheck1[σi]['IRRall'] = experiments.runIRR(M, Reps1[σi], robCheck1[σi]['Baseline'], Δ=.1) # Experiment 3 - applies to all materials
    robCheck1[σi]['RCEff'] = experiments.runRCEff(M, Reps1[σi], robCheck1[σi]['Baseline'], Δα=.1) # Experiment 4
    robCheck1[σi]['taxVirgin'] = experiments.runTaxVirgin(M, Reps1[σi], robCheck1[σi]['Baseline'], Δ=.1) # experiment 5
    robCheck1[σi]['RCMandate'] = experiments.runRCMandate(M, Reps1[σi], robCheck1[σi]['Baseline'], targetRate=.5, m = pd.Index(['Textile'], name = 'm')) # experiment 6
    robCheck1[σi]['taxWasteGen'] = experiments.runTaxWasteGen(M, Reps1[σi], robCheck1[σi]['Baseline'], Δ=.1) # experiment 7
    robCheck1[σi]['subsidyRecycledInputs'] = experiments.runSubsidyRecycledInputs(M, Reps1[σi], robCheck1[σi]['Baseline'], Δ=.1) # experiment 8

Add baseline in the middle:

In [15]:
robCheck1 = list(robCheck1.items())
robCheck1.insert(1, (baseσ, baseShocks))
robCheck1 = dict(robCheck1)

#### A. Results in table format: Change in virgin material

In [16]:
def extractRow_QV(solDict, key, roundTo = 2):
    return [str(round(100 * solDict[key][v]['ΔQvTot_QvTot'].xs(2030), roundTo))  for v in mapExp.values()]

Here is a print of the data input:

In [17]:
for σi in robCheck1:
    print(extractRow_QV(robCheck1, σi)) 

['1.49', '1.32', '0.16', '-0.0', '-7.19', '-2.64', '0.0', '0.06']
['1.4', '1.24', '0.16', '-0.02', '-7.42', '-2.57', '0.0', '0.02']
['1.32', '1.16', '0.15', '-0.04', '-7.65', '-2.5', '0.0', '-0.02']


Here is the data embedded in the table:

In [18]:
def addOneLine_QV(solDict, key, name, eq = '=', par = '\\sigma'):
    return f""" & {name} (${par} {eq} {key}$) & {'& '.join(extractRow_QV(solDict, key))} \\\\
    """    
def printCheck1_QV(solDict, keys, names, eq = '=', par = '\\sigma'):
    return f"""
    \\multirow{{3}}{{*}}{{\parbox{{3cm}}{{Change in virgin material throughput (\\%)}}}}{''.join([addOneLine_QV(solDict, keys[i], names[i], eq = eq, par = par) for i in range(len(keys))])}
"""
print(printCheck1_QV(robCheck1, [round(σi) for σi in robCheck1], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Change in virgin material throughput (\%)}} & Low ($\sigma = 2$) & 1.49& 1.32& 0.16& -0.0& -7.19& -2.64& 0.0& 0.06 \\
     & Base ($\sigma = 3$) & 1.4& 1.24& 0.16& -0.02& -7.42& -2.57& 0.0& 0.02 \\
     & High ($\sigma = 4$) & 1.32& 1.16& 0.15& -0.04& -7.65& -2.5& 0.0& -0.02 \\
    



#### B. Results in table format: Rebound rate

Don't extract for some of these:

In [19]:
noReboundRate = ['taxVirgin','taxWasteGen','subsidyRecycledInputs']

In [20]:
def checkIfRbv(solDict, key, v, roundTo):
    return str(round(solDict[key][v]['RbvTot'].xs(2030), roundTo)) if v not in noReboundRate else "--"
def extractRow_Rbv(solDict, key, roundTo = 2):
    return [checkIfRbv(solDict, key, v, roundTo) for v in mapExp.values()]

Here is a print of the data input:

In [21]:
for σi in robCheck1:
    print(extractRow_Rbv(robCheck1, σi)) 

['1.35', '1.31', '6.34', '1.0', '--', '--', '1.01', '--']
['1.33', '1.29', '6.25', '0.98', '--', '--', '1.01', '--']
['1.31', '1.28', '6.19', '0.96', '--', '--', '1.01', '--']


Here is the data embedded in the table:

In [22]:
def addOneLine_Rbv(solDict, key, name, eq = '=', par = '\\sigma'):
    return f""" & {name} (${par} {eq} {key}$) & {'& '.join(extractRow_Rbv(solDict, key))} \\\\
    """
def printCheck1_Rbv(solDict, keys, names, eq = '=', par = '\\sigma'):
    return f"""
    \\multirow{{3}}{{*}}{{\parbox{{3cm}}{{Rebound rate}}}}{''.join([addOneLine_Rbv(solDict, keys[i], names[i], eq = eq, par = par) for i in range(len(keys))])}
"""
print(printCheck1_Rbv(robCheck1, [round(σi) for σi in robCheck1], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Rebound rate}} & Low ($\sigma = 2$) & 1.35& 1.31& 6.34& 1.0& --& --& 1.01& -- \\
     & Base ($\sigma = 3$) & 1.33& 1.29& 6.25& 0.98& --& --& 1.01& -- \\
     & High ($\sigma = 4$) & 1.31& 1.28& 6.19& 0.96& --& --& 1.01& -- \\
    



#### C. Circularity rate

In [23]:
def extractRow_CR(solDict, key, roundTo = 2):
    return [str(round(100 * solDict[key][v]['CircularRateTot'].xs(2030), roundTo))  for v in mapExp.values()]

Here is a print of the data input:

In [24]:
for σi in robCheck1:
    print(extractRow_CR(robCheck1, σi)) 

['16.17', '16.15', '15.98', '16.15', '17.17', '15.37', '15.96', '16.11']
['16.26', '16.24', '15.98', '16.2', '17.5', '15.28', '15.96', '16.16']
['16.35', '16.32', '15.98', '16.22', '17.81', '15.19', '15.95', '16.2']


Here is the data embedded in the table:

In [25]:
def addOneLine_CR(solDict, key, name, eq = '=', par = '\\sigma'):
    return f""" & {name} (${par} {eq} {key}$) & {'& '.join(extractRow_CR(solDict, key))} \\\\
    """
def printCheck1_RC(solDict, keys, names, eq = '=', par = '\\sigma'):
    return f"""
    \\multirow{{3}}{{*}}{{\parbox{{3cm}}{{Secondary material use rate (\\%)}}}}{''.join([addOneLine_CR(solDict, keys[i], names[i], eq = eq, par = par) for i in range(len(keys))])}
"""
print(printCheck1_RC(robCheck1, [round(σi) for σi in robCheck1], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Secondary material use rate (\%)}} & Low ($\sigma = 2$) & 16.17& 16.15& 15.98& 16.15& 17.17& 15.37& 15.96& 16.11 \\
     & Base ($\sigma = 3$) & 16.26& 16.24& 15.98& 16.2& 17.5& 15.28& 15.96& 16.16 \\
     & High ($\sigma = 4$) & 16.35& 16.32& 15.98& 16.22& 17.81& 15.19& 15.95& 16.2 \\
    



### 2. Substitution between different material types

Revert to main baseline:

In [26]:
[M.db.__setitem__(k, db0[k]) for k in M.db.getTypes(['var'])];

Identify relevant nodes:

In [27]:
m_ZO = m[m.get_level_values('n') == 'ZO'].droplevel('nn').unique() # parent nodes in nesting tree that combines different types of materials

Baseline value:

In [28]:
baseσ = adj.rc_pd(db0('sigma'), m_ZO).mean()

Investigate effect of EOS +/-:

In [29]:
robCheck2 = dict.fromkeys((1e-4, 0.5)) # solution dicts
Reps2 = dict.fromkeys((1e-4, 0.5)) # reporting dicts

Solve baselines:

In [30]:
for σi in robCheck2:
    # Update parameter and re-calibrate:
    M.db.aom(pd.Series(σi, index = m_ZO, name = 'sigma'), priority = 'second')
    M.db.mergeInternal()
    base = M.solve(state = 'C')
    # Add new reporting function:
    Reps2[σi] = report.Standard(base)
    d0i = Reps2[σi](base, {}) # standard report
    d0i['db'] = base # add database
    robCheck2[σi] = {'Baseline': d0i} # add baseline dictionary

For each version, run through experiments and report stuff:

In [31]:
for σi in robCheck2:
    [M.db.__setitem__(k, robCheck2[σi]['Baseline']['db'][k]) for k in M.db.getTypes(['var'])]; # Revert to relevant baseline
    robCheck2[σi]['IRRmetal'] = experiments.runIRR(M, Reps2[σi], robCheck2[σi]['Baseline'], m = pd.Index(['Metal'], name = 'm'), Δ=.1) # Experiment 1
    robCheck2[σi]['IRRplastic'] = experiments.runIRR(M, Reps2[σi], robCheck2[σi]['Baseline'], m = pd.Index(['Plastic'], name = 'm'), Δ=.1) # Experiment 2
    robCheck2[σi]['IRRall'] = experiments.runIRR(M, Reps2[σi], robCheck2[σi]['Baseline'], Δ=.1) # Experiment 3 - applies to all materials
    robCheck2[σi]['RCEff'] = experiments.runRCEff(M, Reps2[σi], robCheck2[σi]['Baseline'], Δα=.1) # Experiment 4
    robCheck2[σi]['taxVirgin'] = experiments.runTaxVirgin(M, Reps2[σi], robCheck2[σi]['Baseline'], Δ=.1) # experiment 5
    robCheck2[σi]['RCMandate'] = experiments.runRCMandate(M, Reps2[σi], robCheck2[σi]['Baseline'], targetRate=.5, m = pd.Index(['Textile'], name = 'm')) # experiment 6
    robCheck2[σi]['taxWasteGen'] = experiments.runTaxWasteGen(M, Reps2[σi], robCheck2[σi]['Baseline'], Δ=.1) # experiment 7
    robCheck2[σi]['subsidyRecycledInputs'] = experiments.runSubsidyRecycledInputs(M, Reps2[σi], robCheck2[σi]['Baseline'], Δ=.1) # experiment 8

Add baseline in the middle:

In [32]:
robCheck2 = list(robCheck2.items())
robCheck2.insert(1, (baseσ, baseShocks))
robCheck2 = dict(robCheck2)

#### A. Results in table format: Change in virgin material

Here is the data embedded in the table:

In [33]:
print(printCheck1_QV(robCheck2, [σi for σi in robCheck2], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Change in virgin material throughput (\%)}} & Low ($\sigma = 0.0001$) & 1.38& 1.21& 0.16& -0.02& -7.43& -2.51& 0.0& 0.02 \\
     & Base ($\sigma = 0.1$) & 1.4& 1.24& 0.16& -0.02& -7.42& -2.57& 0.0& 0.02 \\
     & High ($\sigma = 0.5$) & 1.5& 1.36& 0.13& -0.02& -7.4& -2.81& 0.0& 0.02 \\
    



#### B. Results in table format: Rebound rate

Here is the data embedded in the table:

In [34]:
print(printCheck1_Rbv(robCheck2, [σi for σi in robCheck2], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Rebound rate}} & Low ($\sigma = 0.0001$) & 1.32& 1.29& 6.47& 0.98& --& --& 1.01& -- \\
     & Base ($\sigma = 0.1$) & 1.33& 1.29& 6.25& 0.98& --& --& 1.01& -- \\
     & High ($\sigma = 0.5$) & 1.35& 1.32& 5.37& 0.98& --& --& 1.01& -- \\
    



#### C. Circularity rate

Here is the data embedded in the table:

In [35]:
print(printCheck1_RC(robCheck2, [σi for σi in robCheck2], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Secondary material use rate (\%)}} & Low ($\sigma = 0.0001$) & 16.26& 16.24& 15.98& 16.2& 17.49& 15.29& 15.96& 16.16 \\
     & Base ($\sigma = 0.1$) & 16.26& 16.24& 15.98& 16.2& 17.5& 15.28& 15.96& 16.16 \\
     & High ($\sigma = 0.5$) & 16.28& 16.25& 15.98& 16.19& 17.52& 15.25& 15.96& 16.17 \\
    



### 3. Substitution between material origin (domestic/foreign)

Revert to main baseline:

In [36]:
[M.db.__setitem__(k, db0[k]) for k in M.db.getTypes(['var'])];

Identify relevant nodes:

In [37]:
m_ZOym = m[m.get_level_values('n').isin('RxE_'+M.db('nm_D'))].droplevel('nn').unique() # parent nodes in nesting tree that combines domestic/foreign types

Baseline value:

In [38]:
baseσ = adj.rc_pd(db0('sigma'), m_ZOym).mean()

Investigate effect of EOS +/-:

In [39]:
robCheck3 = dict.fromkeys((2, 5)) # solution dicts
Reps3 = dict.fromkeys((2, 5)) # reporting dicts

Solve baselines:

In [40]:
for σi in robCheck3:
    # Update parameter and re-calibrate:
    M.db.aom(pd.Series(σi, index = m_ZOym, name = 'sigma'), priority = 'second')
    M.db.mergeInternal()
    base = M.solve(state = 'C')
    # Add new reporting function:
    Reps3[σi] = report.Standard(base)
    d0i = Reps3[σi](base, {}) # standard report
    d0i['db'] = base # add database
    robCheck3[σi] = {'Baseline': d0i} # add baseline dictionary

For each version, run through experiments and report stuff:

In [41]:
for σi in robCheck3:
    [M.db.__setitem__(k, robCheck3[σi]['Baseline']['db'][k]) for k in M.db.getTypes(['var'])]; # Revert to relevant baseline
    robCheck3[σi]['IRRmetal'] = experiments.runIRR(M, Reps3[σi], robCheck3[σi]['Baseline'], m = pd.Index(['Metal'], name = 'm'), Δ=.1) # Experiment 1
    robCheck3[σi]['IRRplastic'] = experiments.runIRR(M, Reps3[σi], robCheck3[σi]['Baseline'], m = pd.Index(['Plastic'], name = 'm'), Δ=.1) # Experiment 2
    robCheck3[σi]['IRRall'] = experiments.runIRR(M, Reps3[σi], robCheck3[σi]['Baseline'], Δ=.1) # Experiment 3 - applies to all materials
    robCheck3[σi]['RCEff'] = experiments.runRCEff(M, Reps3[σi], robCheck3[σi]['Baseline'], Δα=.1) # Experiment 4
    robCheck3[σi]['taxVirgin'] = experiments.runTaxVirgin(M, Reps3[σi], robCheck3[σi]['Baseline'], Δ=.1) # experiment 5
    robCheck3[σi]['RCMandate'] = experiments.runRCMandate(M, Reps3[σi], robCheck3[σi]['Baseline'], targetRate=.5, m = pd.Index(['Textile'], name = 'm')) # experiment 6
    robCheck3[σi]['taxWasteGen'] = experiments.runTaxWasteGen(M, Reps3[σi], robCheck3[σi]['Baseline'], Δ=.1) # experiment 7
    robCheck3[σi]['subsidyRecycledInputs'] = experiments.runSubsidyRecycledInputs(M, Reps3[σi], robCheck3[σi]['Baseline'], Δ=.1) # experiment 8

Add baseline in the middle:

In [42]:
robCheck3 = list(robCheck3.items())
robCheck3.insert(1, (baseσ, baseShocks))
robCheck3 = dict(robCheck3)

#### A. Results in table format: Change in virgin material

Here is the data embedded in the table:

In [43]:
print(printCheck1_QV(robCheck3, [round(σi) for σi in robCheck3], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Change in virgin material throughput (\%)}} & Low ($\sigma = 2$) & 1.4& 1.24& 0.16& -0.03& -7.35& -2.57& 0.0& 0.02 \\
     & Base ($\sigma = 3$) & 1.4& 1.24& 0.16& -0.02& -7.42& -2.57& 0.0& 0.02 \\
     & High ($\sigma = 5$) & 1.41& 1.24& 0.16& -0.02& -7.52& -2.58& 0.0& 0.01 \\
    



#### B. Results in table format: Rebound rate

Here is the data embedded in the table:

In [44]:
print(printCheck1_Rbv(robCheck3, [round(σi) for σi in robCheck3], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Rebound rate}} & Low ($\sigma = 2$) & 1.33& 1.29& 6.24& 0.98& --& --& 1.01& -- \\
     & Base ($\sigma = 3$) & 1.33& 1.29& 6.25& 0.98& --& --& 1.01& -- \\
     & High ($\sigma = 5$) & 1.33& 1.29& 6.27& 0.98& --& --& 1.01& -- \\
    



#### C. Circularity rate

Here is the data embedded in the table:

In [45]:
print(printCheck1_RC(robCheck3, [round(σi) for σi in robCheck3], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Secondary material use rate (\%)}} & Low ($\sigma = 2$) & 16.27& 16.25& 15.99& 16.21& 17.48& 15.29& 15.97& 16.15 \\
     & Base ($\sigma = 3$) & 16.26& 16.24& 15.98& 16.2& 17.5& 15.28& 15.96& 16.16 \\
     & High ($\sigma = 5$) & 16.25& 16.22& 15.97& 16.17& 17.53& 15.27& 15.94& 16.17 \\
    



### 4. Substitution between materials and services

Revert to main baseline:

In [46]:
[M.db.__setitem__(k, db0[k]) for k in M.db.getTypes(['var'])];

Identify relevant nodes:

In [47]:
m_ZOY = m[m.get_level_values('n') == 'RxE'].droplevel('nn').unique() # parent nodes in nesting tree that combines domestic/foreign types

Baseline value:

In [48]:
baseσ = round(adj.rc_pd(db0('sigma'), m_ZOY).mean(), 3)

Investigate effect of EOS +/-:

In [49]:
robCheck4 = dict.fromkeys((.35, .95)) # solution dicts
Reps4 = dict.fromkeys((.35, .95)) # reporting dicts

Solve baselines:

In [50]:
for σi in robCheck4:
    # Update parameter and re-calibrate:
    M.db.aom(pd.Series(σi, index = m_ZOY, name = 'sigma'), priority = 'second')
    M.db.mergeInternal()
    base = M.solve(state = 'C')
    # Add new reporting function:
    Reps4[σi] = report.Standard(base)
    d0i = Reps4[σi](base, {}) # standard report
    d0i['db'] = base # add database
    robCheck4[σi] = {'Baseline': d0i} # add baseline dictionary

For each version, run through experiments and report stuff:

In [51]:
for σi in robCheck4:
    [M.db.__setitem__(k, robCheck4[σi]['Baseline']['db'][k]) for k in M.db.getTypes(['var'])]; # Revert to relevant baseline
    robCheck4[σi]['IRRmetal'] = experiments.runIRR(M, Reps4[σi], robCheck4[σi]['Baseline'], m = pd.Index(['Metal'], name = 'm'), Δ=.1) # Experiment 1
    robCheck4[σi]['IRRplastic'] = experiments.runIRR(M, Reps4[σi], robCheck4[σi]['Baseline'], m = pd.Index(['Plastic'], name = 'm'), Δ=.1) # Experiment 2
    robCheck4[σi]['IRRall'] = experiments.runIRR(M, Reps4[σi], robCheck4[σi]['Baseline'], Δ=.1) # Experiment 3 - applies to all materials
    robCheck4[σi]['RCEff'] = experiments.runRCEff(M, Reps4[σi], robCheck4[σi]['Baseline'], Δα=.1) # Experiment 4
    robCheck4[σi]['taxVirgin'] = experiments.runTaxVirgin(M, Reps4[σi], robCheck4[σi]['Baseline'], Δ=.1) # experiment 5
    robCheck4[σi]['RCMandate'] = experiments.runRCMandate(M, Reps4[σi], robCheck4[σi]['Baseline'], targetRate=.5, m = pd.Index(['Textile'], name = 'm')) # experiment 6
    robCheck4[σi]['taxWasteGen'] = experiments.runTaxWasteGen(M, Reps4[σi], robCheck4[σi]['Baseline'], Δ=.1) # experiment 7
    robCheck4[σi]['subsidyRecycledInputs'] = experiments.runSubsidyRecycledInputs(M, Reps4[σi], robCheck4[σi]['Baseline'], Δ=.1) # experiment 8

Add baseline in the middle:

In [52]:
robCheck4 = list(robCheck4.items())
robCheck4.insert(1, (baseσ, baseShocks))
robCheck4 = dict(robCheck4)

#### A. Results in table format: Change in virgin material

Here is the data embedded in the table:

In [53]:
print(printCheck1_QV(robCheck4, [σi for σi in robCheck4], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Change in virgin material throughput (\%)}} & Low ($\sigma = 0.35$) & 1.02& 0.89& 0.13& -0.03& -6.11& -1.85& 0.0& -0.0 \\
     & Base ($\sigma = 0.64$) & 1.4& 1.24& 0.16& -0.02& -7.42& -2.57& 0.0& 0.02 \\
     & High ($\sigma = 0.95$) & 1.81& 1.61& 0.19& -0.01& -8.78& -3.32& 0.0& 0.04 \\
    



#### B. Results in table format: Rebound rate

Here is the data embedded in the table:

In [54]:
print(printCheck1_Rbv(robCheck4, [σi for σi in robCheck4], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Rebound rate}} & Low ($\sigma = 0.35$) & 1.24& 1.21& 5.25& 0.97& --& --& 1.01& -- \\
     & Base ($\sigma = 0.64$) & 1.33& 1.29& 6.25& 0.98& --& --& 1.01& -- \\
     & High ($\sigma = 0.95$) & 1.43& 1.38& 7.3& 0.99& --& --& 1.01& -- \\
    



#### C. Circularity rate

Here is the data embedded in the table:

In [55]:
print(printCheck1_RC(robCheck4, [σi for σi in robCheck4], ['Low', 'Base', 'High']))


    \multirow{3}{*}{\parbox{3cm}{Secondary material use rate (\%)}} & Low ($\sigma = 0.35$) & 16.25& 16.22& 15.99& 16.2& 17.4& 15.48& 15.96& 16.15 \\
     & Base ($\sigma = 0.64$) & 16.26& 16.24& 15.98& 16.2& 17.5& 15.28& 15.96& 16.16 \\
     & High ($\sigma = 0.95$) & 16.28& 16.26& 15.98& 16.19& 17.6& 15.08& 15.95& 16.17 \\
    



### 5. Assumption on internal recycling efficiency 

The current robustness check is carried out slightly differently. The reason is that the robustness check occurs by changing an assumption early in the data-processing step. Thus, instead of carefully going through the model and where this changes things and imposing them here as a shock, we instead run notebooks `A1-A3` with different internal rates of efficiency and store the calibrated models. This has to be done before the following works:

Load the other models with low/high internal efficiency assumption:

In [56]:
_names = {'-10\mbox{{ }}p.p.': 'vRbstA_L', '+10\mbox{{ }}p.p.': 'vRbstA_H'}
M_IRE = {k: mCGE.WasteManagementCGE.load(os.path.join(d['data'], f'{_names[k]}{t0}CGE')) for k in _names}

For each of the model versions, apply the shocks:

In [57]:
robCheck5 = dict.fromkeys(_names) # solution dicts
Reps5 = dict.fromkeys(_names) # reporting dicts

Baselines are already solved for here, so start by creating structures similar to other robustness checks:

In [58]:
for k, Mi in M_IRE.items():
    base = Mi.db.copy()
    Reps5[k] = report.Standard(base)
    d0i = Reps5[k](base, {})
    d0i['db'] = base
    robCheck5[k] = {'Baseline': d0i}

For each version, run through experiments and report stuff:

In [59]:
for k, Mi in M_IRE.items():
    robCheck5[k]['IRRmetal'] = experiments.runIRR(Mi, Reps5[k], robCheck5[k]['Baseline'], m = pd.Index(['Metal'], name = 'm'), Δ=.1) # Experiment 1
    robCheck5[k]['IRRplastic'] = experiments.runIRR(Mi, Reps5[k], robCheck5[k]['Baseline'], m = pd.Index(['Plastic'], name = 'm'), Δ=.1) # Experiment 2
    robCheck5[k]['IRRall'] = experiments.runIRR(Mi, Reps5[k], robCheck5[k]['Baseline'], Δ=.1) # Experiment 3 - applies to all materials
    robCheck5[k]['RCEff'] = experiments.runRCEff(Mi, Reps5[k], robCheck5[k]['Baseline'], Δα=.1) # Experiment 4
    robCheck5[k]['taxVirgin'] = experiments.runTaxVirgin(Mi, Reps5[k], robCheck5[k]['Baseline'], Δ=.1) # experiment 5
    robCheck5[k]['RCMandate'] = experiments.runRCMandate(Mi, Reps5[k], robCheck5[k]['Baseline'], targetRate=.5, m = pd.Index(['Textile'], name = 'm')) # experiment 6
    robCheck5[k]['taxWasteGen'] = experiments.runTaxWasteGen(Mi, Reps5[k], robCheck5[k]['Baseline'], Δ=.1) # experiment 7
    robCheck5[k]['subsidyRecycledInputs'] = experiments.runSubsidyRecycledInputs(Mi, Reps5[k], robCheck5[k]['Baseline'], Δ=.1) # experiment 8

Add baseline in the middle:

In [60]:
robCheck5 = list(robCheck5.items())
robCheck5.insert(1, ('-', baseShocks))
robCheck5 = dict(robCheck5)

#### A. Results in table format: Change in virgin material

In [61]:
print(printCheck1_QV(robCheck5, [k for k in robCheck5], ['Low', 'Base', 'High'], eq = '', par = ''))


    \multirow{3}{*}{\parbox{3cm}{Change in virgin material throughput (\%)}} & Low ($  -10\mbox{{ }}p.p.$) & 1.72& 1.48& 0.24& -0.02& -7.42& -2.57& 0.0& 0.02 \\
     & Base ($  -$) & 1.4& 1.24& 0.16& -0.02& -7.42& -2.57& 0.0& 0.02 \\
     & High ($  +10\mbox{{ }}p.p.$) & 1.21& 1.07& 0.14& -0.02& -7.42& -2.57& 0.0& 0.02 \\
    



#### B. Results in table format: Rebound rate

Here is the data embedded in the table:

In [62]:
print(printCheck1_Rbv(robCheck5, [σi for σi in robCheck5], ['Low', 'Base', 'High'],  eq = '', par = ''))


    \multirow{3}{*}{\parbox{3cm}{Rebound rate}} & Low ($  -10\mbox{{ }}p.p.$) & 1.38& 1.33& 5.68& 0.98& --& --& 1.01& -- \\
     & Base ($  -$) & 1.33& 1.29& 6.25& 0.98& --& --& 1.01& -- \\
     & High ($  +10\mbox{{ }}p.p.$) & 1.3& 1.26& 6.55& 0.98& --& --& 1.01& -- \\
    



#### C. Circularity rate

Here is the data embedded in the table:

In [63]:
print(printCheck1_RC(robCheck5, [σi for σi in robCheck5], ['Low', 'Base', 'High'], eq = '', par = ''))


    \multirow{3}{*}{\parbox{3cm}{Secondary material use rate (\%)}} & Low ($  -10\mbox{{ }}p.p.$) & 16.3& 16.25& 16.01& 16.2& 17.5& 15.28& 15.96& 16.16 \\
     & Base ($  -$) & 16.26& 16.24& 15.98& 16.2& 17.5& 15.28& 15.96& 16.16 \\
     & High ($  +10\mbox{{ }}p.p.$) & 16.25& 16.23& 15.98& 16.2& 17.5& 15.28& 15.96& 16.16 \\
    



### 6. Assumptions on waste generating intensities

The robustness check similarly involves altering assumptions early in the data processing. We run notebooks `B1-B3` beforehand with the uniform waste generating intensity; This has to be done before the following works.  

Load the other model with uniform waste generating intensity assumption:

In [64]:
_names = {'Uniform': 'vRbstB'}
M_RbstB = {k: mCGE.WasteManagementCGE.load(os.path.join(d['data'], f'{_names[k]}{t0}CGE')) for k in _names}

For each of the model versions, apply the shocks:

In [65]:
robCheck6 = dict.fromkeys(_names) # solution dicts
Reps6 = dict.fromkeys(_names) # reporting dicts

Baselines are already solved for here, so start by creating structures similar to other robustness checks:

In [66]:
for k, Mi in M_RbstB.items():
    base = Mi.db.copy()
    Reps6[k] = report.Standard(base)
    d0i = Reps6[k](base, {})
    d0i['db'] = base
    robCheck6[k] = {'Baseline': d0i}

For each version, run through experiments and report stuff:

In [67]:
for k, Mi in M_RbstB.items():
    robCheck6[k]['IRRmetal'] = experiments.runIRR(Mi, Reps6[k], robCheck6[k]['Baseline'], m = pd.Index(['Metal'], name = 'm'), Δ=.1) # Experiment 1
    robCheck6[k]['IRRplastic'] = experiments.runIRR(Mi, Reps6[k], robCheck6[k]['Baseline'], m = pd.Index(['Plastic'], name = 'm'), Δ=.1) # Experiment 2
    robCheck6[k]['IRRall'] = experiments.runIRR(Mi, Reps6[k], robCheck6[k]['Baseline'], Δ=.1) # Experiment 3 - applies to all materials
    robCheck6[k]['RCEff'] = experiments.runRCEff(Mi, Reps6[k], robCheck6[k]['Baseline'], Δα=.1) # Experiment 4
    robCheck6[k]['taxVirgin'] = experiments.runTaxVirgin(Mi, Reps6[k], robCheck6[k]['Baseline'], Δ=.1) # experiment 5
    robCheck6[k]['RCMandate'] = experiments.runRCMandate(Mi, Reps6[k], robCheck6[k]['Baseline'], targetRate=.5, m = pd.Index(['Textile'], name = 'm')) # experiment 6
    robCheck6[k]['taxWasteGen'] = experiments.runTaxWasteGen(Mi, Reps6[k], robCheck6[k]['Baseline'], Δ=.1) # experiment 7
    robCheck6[k]['subsidyRecycledInputs'] = experiments.runSubsidyRecycledInputs(Mi, Reps6[k], robCheck6[k]['Baseline'], Δ=.1) # experiment 8

Add baseline in the middle:

In [68]:
robCheck6 = list(robCheck6.items())
robCheck6.insert(0, ('Baseline', baseShocks))
robCheck6 = dict(robCheck6)

*New printing functions for this case where we simply have a "uniform" and "baseline" scenario.*

#### A. Results in table format: Change in virgin material

In [69]:
def addOneLine6_QV(solDict, key, name):
    return f""" & {name} & {'& '.join(extractRow_QV(solDict, key))} \\\\
    """    
def printCheck6_QV(solDict, keys):
    return f"""
    \\multirow{{2}}{{*}}{{\parbox{{3cm}}{{Change in virgin material throughput (\\%)}}}}{''.join([addOneLine6_QV(solDict, keys[i], keys[i]) for i in range(len(keys))])}
"""
print(printCheck6_QV(robCheck6, list(robCheck6)))    


    \multirow{2}{*}{\parbox{3cm}{Change in virgin material throughput (\%)}} & Baseline & 1.4& 1.24& 0.16& -0.02& -7.42& -2.57& 0.0& 0.02 \\
     & Uniform & 1.4& 1.24& 0.16& -0.02& -7.42& -2.57& 0.0& 0.02 \\
    



#### B. Results in table format: Rebound rate

In [88]:
def addOneLine6_Rbv(solDict, key, name):
    return f""" & {name} & {'& '.join(extractRow_Rbv(solDict, key))} \\\\
    """
def printCheck6_Rbv(solDict, keys):
    return f"""
    \\multirow{{2}}{{*}}{{\parbox{{3cm}}{{Rebound rate}}}}{''.join([addOneLine6_Rbv(solDict, keys[i], keys[i]) for i in range(len(keys))])}
"""
print(printCheck6_Rbv(robCheck6, list(robCheck6)))    


    \multirow{2}{*}{\parbox{3cm}{Rebound rate}} & Baseline & 1.33& 1.29& 6.25& 0.98& --& --& 1.01& -- \\
     & Uniform & 1.33& 1.29& 6.25& 0.98& --& --& 1.01& -- \\
    



#### C. Circularity rate

In [89]:
def addOneLine6_CR(solDict, key, name):
    return f""" & {name} & {'& '.join(extractRow_CR(solDict, key))} \\\\
    """
def printCheck6_RC(solDict, keys):
    return f"""
    \\multirow{{2}}{{*}}{{\parbox{{3cm}}{{Secondary material use rate (\\%)}}}}{''.join([addOneLine6_CR(solDict, keys[i], keys[i]) for i in range(len(keys))])}
"""
print(printCheck6_RC(robCheck6, list(robCheck6)))    


    \multirow{2}{*}{\parbox{3cm}{Secondary material use rate (\%)}} & Baseline & 16.26& 16.24& 15.98& 16.2& 17.5& 15.28& 15.96& 16.16 \\
     & Uniform & 16.26& 16.24& 15.98& 16.2& 17.5& 15.28& 15.96& 16.16 \\
    



### 7. Assumption on default $\beta_m$ in recycling technology

Robustness check similar to `5` regarding different assumptions in internal recycling efficiency. Requires running notebooks `C1-C3` with varying levels of $\beta_m$. 

Load:

In [90]:
_names = {'0.5': 'vRbstC_L', '1.5': 'vRbstC_H'}
M_RT = {k: mCGE.WasteManagementCGE.load(os.path.join(d['data'], f'{_names[k]}{t0}CGE')) for k in _names}

For each of the model versions, apply the shocks:

In [91]:
robCheck7 = dict.fromkeys(_names) # solution dicts
Reps7 = dict.fromkeys(_names) # reporting dicts

Baselines are already solved for here, so start by creating structures similar to other robustness checks:

In [92]:
for k, Mi in M_RT.items():
    base = Mi.db.copy()
    Reps7[k] = report.Standard(base)
    d0i = Reps7[k](base, {})
    d0i['db'] = base
    robCheck7[k] = {'Baseline': d0i}

For each version, run through experiments and report stuff:

In [93]:
for k, Mi in M_RT.items():
    robCheck7[k]['IRRmetal'] = experiments.runIRR(Mi, Reps7[k], robCheck7[k]['Baseline'], m = pd.Index(['Metal'], name = 'm'), Δ=.1) # Experiment 1
    robCheck7[k]['IRRplastic'] = experiments.runIRR(Mi, Reps7[k], robCheck7[k]['Baseline'], m = pd.Index(['Plastic'], name = 'm'), Δ=.1) # Experiment 2
    robCheck7[k]['IRRall'] = experiments.runIRR(Mi, Reps7[k], robCheck7[k]['Baseline'], Δ=.1) # Experiment 3 - applies to all materials
    robCheck7[k]['RCEff'] = experiments.runRCEff(Mi, Reps7[k], robCheck7[k]['Baseline'], Δα=.1) # Experiment 4
    robCheck7[k]['taxVirgin'] = experiments.runTaxVirgin(Mi, Reps7[k], robCheck7[k]['Baseline'], Δ=.1) # experiment 5
    robCheck7[k]['RCMandate'] = experiments.runRCMandate(Mi, Reps7[k], robCheck7[k]['Baseline'], targetRate=.5, m = pd.Index(['Textile'], name = 'm')) # experiment 6
    robCheck7[k]['taxWasteGen'] = experiments.runTaxWasteGen(Mi, Reps7[k], robCheck7[k]['Baseline'], Δ=.1) # experiment 7
    robCheck7[k]['subsidyRecycledInputs'] = experiments.runSubsidyRecycledInputs(Mi, Reps7[k], robCheck7[k]['Baseline'], Δ=.1) # experiment 8

Add baseline in the middle:

In [94]:
robCheck7 = list(robCheck7.items())
robCheck7.insert(1, ('1.0', baseShocks))
robCheck7 = dict(robCheck7)

#### A. Results in table format: Change in virgin material

In [95]:
print(printCheck1_QV(robCheck7, [k for k in robCheck7], ['Low', 'Base', 'High'], eq = '', par = ''))


    \multirow{3}{*}{\parbox{3cm}{Change in virgin material throughput (\%)}} & Low ($  0.5$) & 1.4& 1.24& 0.16& -0.03& -7.42& -2.57& 0.0& 0.02 \\
     & Base ($  1.0$) & 1.4& 1.24& 0.16& -0.02& -7.42& -2.57& 0.0& 0.02 \\
     & High ($  1.5$) & 1.4& 1.24& 0.16& -0.02& -7.42& -2.57& 0.0& 0.02 \\
    



#### B. Results in table format: Rebound rate

Here is the data embedded in the table:

In [96]:
print(printCheck1_Rbv(robCheck7, [σi for σi in robCheck7], ['Low', 'Base', 'High'],  eq = '', par = ''))


    \multirow{3}{*}{\parbox{3cm}{Rebound rate}} & Low ($  0.5$) & 1.33& 1.29& 6.25& 0.97& --& --& 1.0& -- \\
     & Base ($  1.0$) & 1.33& 1.29& 6.25& 0.98& --& --& 1.01& -- \\
     & High ($  1.5$) & 1.33& 1.29& 6.25& 0.98& --& --& 1.01& -- \\
    



#### C. Circularity rate

Here is the data embedded in the table:

In [97]:
print(printCheck1_RC(robCheck7, [σi for σi in robCheck7], ['Low', 'Base', 'High'], eq = '', par = ''))


    \multirow{3}{*}{\parbox{3cm}{Secondary material use rate (\%)}} & Low ($  0.5$) & 16.26& 16.24& 15.98& 16.26& 17.5& 15.28& 15.96& 16.16 \\
     & Base ($  1.0$) & 16.26& 16.24& 15.98& 16.2& 17.5& 15.28& 15.96& 16.16 \\
     & High ($  1.5$) & 16.26& 16.24& 15.98& 16.15& 17.5& 15.28& 15.96& 16.16 \\
    

